In [0]:
from pyspark.sql import functions as f

In [0]:
%sql
use catalog skylens_macro;
use schema team1;

In [0]:
# BASE PATH 
BASE_PATH = "/Volumes/Skylens_macro/team1/preprocess_data"  

# Bronze output path
BRONZE_PATH = "/Volumes/Skylens_macro/team1/bronze"  

# Checkpoints path
CHECKPOINTS_PATH = "/Volumes/Skylens_macro/team1/checkpoints/bronze"   

In [0]:
%sql
use catalog Skylens_macro;
use schema Skylens_macro.team1;

In [0]:
import uuid

df_bronze_flights = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINTS_PATH}/flights/schema/")
    .load(f"{BASE_PATH}/flights/")
)

df_bronze_flights = df_bronze_flights \
    .withColumn("ingestion_timestamp", f.current_timestamp()) \
    .withColumn("source_file_name", f.lit('flights')) \
    .withColumn("batch_id", f.lit(str(uuid.uuid4()))) \
    .withColumn(
        "source_month",
       df_bronze_flights['MONTH'].cast('string')
    
    )



querey=(
    df_bronze_flights \
     .writeStream\
     .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{CHECKPOINTS_PATH}/flights/checkpoints/") \
    .partitionBy("source_month") \
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable('Skylens_macro.team1.bronze_flights')
)

querey.awaitTermination()


In [0]:
df=spark.read.csv('/Volumes/skylens_macro/team1/preprocess_data/airports/',header=True,inferSchema=True)

df.show(2)

In [0]:
import uuid

df_bronze_airports = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", f"{CHECKPOINTS_PATH}/airports/schema/")
    .load(f"{BASE_PATH}/airports/")
    
)

df_bronze_airports = df_bronze_airports \
    .withColumn("ingestion_timestamp", f.current_timestamp()) \
    .withColumn("source_file_name", f.lit('airports')) \
    .withColumn("batch_id", f.lit(str(uuid.uuid4())))
    



querey=(
    df_bronze_airports \
     .writeStream\
     .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{CHECKPOINTS_PATH}/airports/checkpoints/")\
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable('Skylens_macro.team1.bronze_airports')
)

querey.awaitTermination()


In [0]:
# import uuid
# file_name=['airports','airlines']
# queries=[]


# for file in file_name:

#     if file == "airlines":
#         header_flag = "false"
       
#     else:
#         header_flag = "true"

#     df_bronze = (
#     spark.readStream
#     .format("cloudFiles")
#     .option("cloudFiles.format", "csv")
#     .option("header", header_flag)
#     .option("cloudFiles.inferColumnTypes", "true")
#     .option("cloudFiles.schemaLocation", f"{CHECKPOINTS_PATH}/{file}/schema/")
#     .load(f"{BASE_S3_PATH}/{file}/")
# )
#     if file == "airlines":
#          df_bronze = df_bronze.select(
#             f.col(df_bronze.columns[0]).alias("iata_code"),
#             f.col(df_bronze.columns[1]).alias("airline_name")
#         )
       
       

       
        

#     df_bronze = (df_bronze \
#     .withColumn("ingestion_timestamp", f.current_timestamp()) \
#     .withColumn("source_file_name", f.lit(f"{file}")) \
#     .withColumn("batch_id", f.lit(str(uuid.uuid4()))) \
    
#     )



#     querey=(
#     df_bronze \
#      .writeStream\
#      .format("delta") \
#     .outputMode("append") \
#     .option("checkpointLocation", f"{CHECKPOINTS_PATH}/{file}/checkpoints/") \
#         .option("path",f"{BRONZE_PATH}/{file}/")
#         .option("mergeSchema", "true")
#         .trigger(availableNow=True)\
#         .toTable(f"bronze_{file}")
#         )
    
#     queries.append(querey)
    
# for query in queries:
#     query.awaitTermination()





In [0]:
%sql
drop table if exists bronze_airlines;

In [0]:
import uuid
from pyspark.sql.types import StructType, StructField, StringType

file_name = ['airlines']
queries = []


# airlines_schema = StructType([
#     StructField("iata_code", StringType(), True),
#     StructField("airline_name", StringType(), True)
# ])

for file in file_name:

    if file == "airlines":
        df_bronze = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")        
            .option("cloudFiles.schemaLocation", f"{CHECKPOINTS_PATH}/{file}/schema/")
            .load(f"{BASE_PATH}/{file}/")
        )
        
        

    else:
        df_bronze = (
            spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("header", "true")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaLocation", f"{CHECKPOINTS_PATH}/{file}/schema/")
            .load(f"{BASE_PATH}/{file}/")
        )

    df_bronze = (
        df_bronze
        .withColumn("ingestion_timestamp", f.current_timestamp())
        .withColumn("source_file_name", f.lit(f"{file}"))
        .withColumn("batch_id", f.lit(str(uuid.uuid4())))
    )

    query = (
        df_bronze
        .writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{CHECKPOINTS_PATH}/{file}/checkpoints/")
        .option("mergeSchema", "true")
        .trigger(availableNow=True)
        .toTable(f"Skylens_macro.team1.bronze_{file}")
    )

    queries.append(query)

for query in queries:
    query.awaitTermination()

In [0]:
df=spark.read.csv(f"{BASE_PATH}/airlines/",header=True,inferSchema=True)
df.show(truncate=False)

In [0]:
df=spark.read.csv(f"{BASE_PATH}/airlines/",header=True,inferSchema=True)

df.show(10,truncate=False)

In [0]:
%sql
show tables;

In [0]:
%sql
select * from skylens_macro.team1.bronze_airlines;